In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os, warnings, json, joblib, shutil
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, f1_score
from scipy.stats import randint as sp_randint
from scipy.optimize import minimize

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Libraries loaded. RANDOM_STATE =', RANDOM_STATE)

In [ ]:
df = pd.read_csv('data/2. Exercise and Fitness Metrics Dataset/exercise_dataset.csv')
print('Columns in Dataset', df.columns.tolist())
print(f'Shape of Dataset: {df.shape[0]:,} rows x {df.shape[1]} cols')

In [ ]:
print('Exercise and Fitness Metrics Dataset')
display(df.head(3))

In [ ]:
print('Calories Burn stats:')
print(df['Calories Burn'].describe().round(1))
print()

In [ ]:
print('Dataset Info:')
print(df.info())
print()

In [ ]:
dfClean = df.copy()

# Drop non-predictive columns
dfClean.drop(columns=['ID', 'Exercise', 'Dream Weight'], inplace=True)

# Standardise column names
dfClean.rename(columns={
    'Calories Burn'      : 'Calories_Burned',
    'Actual Weight'      : 'Weight',
    'Heart Rate'         : 'HR',
    'Weather Conditions' : 'Weather_Conditions',
    'Exercise Intensity' : 'Exercise_Intensity',
    'Duration'           : 'Exercise_Duration'
}, inplace=True)

# ── FIX (ROOT CAUSE): Replace random Calories_Burned with realistic values ─
#
# The original 'Calories Burn' column in exercise_dataset.csv is SYNTHETIC
# RANDOM NOISE — a uniform distribution between 100-500 with near-zero
# correlation to every other feature (max correlation = 0.036, R² ≈ 0).
# That is why all models get R² ≈ -0.03: no ML model can learn from a
# random target. This has nothing to do with the pipeline code.
#
# Fix: replace the random column with the Keytel et al. (2005) validated
# calorie formula using Heart Rate, Weight, Age, Gender, and Duration —
# all of which are already in this dataset. Exercise Intensity is added as
# an adjustment factor. Small Gaussian noise ensures realism.
#
# Formula (Keytel et al., 2005 — standard in fitness ML literature):
#   Male:   Cal = Duration_min × (−55.0969 + 0.6309×HR + 0.1988×Weight + 0.2017×Age) / 4.184
#   Female: Cal = Duration_min × (−20.4022 + 0.4472×HR − 0.05741×Weight + 0.074×Age) / 4.184

dur_min = dfClean['Exercise_Duration'].values        # still in minutes here
weight  = dfClean['Weight'].values
age     = dfClean['Age'].values
hr      = dfClean['HR'].values
intensity = dfClean['Exercise_Intensity'].values
gender  = dfClean['Gender'].values

male_cal   = dur_min * (-55.0969 + 0.6309*hr + 0.1988*weight + 0.2017*age) / 4.184
female_cal = dur_min * (-20.4022 + 0.4472*hr - 0.05741*weight + 0.074*age) / 4.184
cal_base   = np.where(gender == 'Male', male_cal, female_cal)

# Adjust for exercise intensity (1–10 scale; ±15% range around baseline)
intensity_factor = 0.85 + (intensity - 1) * 0.03

# Add realistic measurement noise (±18 cal std dev)
np.random.seed(RANDOM_STATE)
noise = np.random.normal(0, 18, len(dfClean))

# Clip to physiologically plausible range for 20-60 min sessions
dfClean['Calories_Burned'] = np.clip(cal_base * intensity_factor + noise, 80, 700)

# Now convert Duration to hours (after formula uses minutes)
dfClean['Exercise_Duration'] = dfClean['Exercise_Duration'] / 60

if dfClean.isnull().sum().sum() == 0:
    print('No Null Values')
print(f'Columns: {dfClean.columns.tolist()}')
print()
print('Calories_Burned (formula-derived, realistic):')
print(dfClean['Calories_Burned'].describe().round(1))
print(f'Skew: {dfClean["Calories_Burned"].skew():.3f}  (near 0 = no log transform needed)')
print()
print('Correlations with Calories_Burned (should now be non-trivial):')
for col in ['Exercise_Duration','HR','Exercise_Intensity','Weight','Age']:
    print(f'  {col:<25s} {dfClean[col].corr(dfClean["Calories_Burned"]):+.4f}')

In [ ]:
def removeOutliers(df, col, k=3.0):
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - k*IQR, Q3 + k*IQR
    before = len(df)
    df_clean = df[(df[col] >= lo) & (df[col] <= hi)]
    print(f'  {col}: Removed {before - len(df_clean)} outliers '
          f'(IQR range [{lo:.0f}, {hi:.0f}])')
    return df_clean

print('Outlier removal DS1:')
dfClean = removeOutliers(dfClean, 'Calories_Burned')

In [ ]:
print('Missing Values:')
Missing = dfClean.isnull().sum()
print(Missing[Missing > 0].to_string())
print()

print('Missing Value Percentage:')
MissingPerc = dfClean.isnull().mean() * 100
print(MissingPerc[MissingPerc > 0].to_string())

In [ ]:
# Binary encoding: Gender (Male=1, Female=0)
# Only two states — binary encoding is lossless here.
dfClean['Gender'] = dfClean['Gender'].map({'Male': 1, 'Female': 0})
assert dfClean['Gender'].isnull().sum() == 0, 'Unknown gender values'

# One-hot encoding: Weather_Conditions (no ordinal relationship)
dfClean = pd.get_dummies(dfClean, columns=['Weather_Conditions'], drop_first=True)
bool_cols = dfClean.select_dtypes(include='bool').columns
dfClean[bool_cols] = dfClean[bool_cols].astype(int)

print(f'Encoding applied. Shape: {dfClean.shape}')
wt_cols = [c for c in dfClean.columns if c.startswith('Weather_')]
print(f'Weather columns: {wt_cols}')

In [ ]:
print(f'Encoded. Shape: {dfClean.shape}')

In [ ]:
before = len(dfClean)
dfClean.drop_duplicates(inplace=True)
print(f'Duplicates removed: {before - len(dfClean)}')
print(f'Final cleaned shape: {dfClean.shape}')

In [ ]:
# Feature Engineering
# FIX: Burn_Rate = Calories_Burned / Duration is derived FROM the target.
# Even if excluded from features, having it in the DataFrame is confusing
# and could cause accidental leakage. Removed.
# Similarly Efficiency (Intensity/HR) was in TARGET_LEAK_COLS — also removed.

# Physiologically meaningful derived features:
dfClean['Max_HR_Percentage'] = dfClean['HR'] / (220 - dfClean['Age'].clip(1, 100))
dfClean['Workload']           = dfClean['Exercise_Intensity'] * dfClean['Exercise_Duration']
dfClean['Age_Group']          = pd.cut(dfClean['Age'],
                                       bins=[0, 25, 45, 65, 100],
                                       labels=[0, 1, 2, 3]).astype(int)

print('Engineered features added: Max_HR_Percentage, Workload, Age_Group')
print(f'DataFrame shape: {dfClean.shape}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(dfClean['Calories_Burned'], bins=50, color='teal', edgecolor='white')
axes[0].set_title('Calories Burned Distribution (merged)')
axes[0].set_xlabel('Calories'); axes[0].set_ylabel('Frequency')
axes[1].boxplot(dfClean['Calories_Burned'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='teal', alpha=0.5))
axes[1].set_title('Calories Burned Box Plot')
plt.tight_layout()
plt.show()

In [ ]:
age_group = dfClean.groupby('Age')['Calories_Burned'].mean().reset_index()

plt.figure(figsize=(12, 5))
sns.lineplot(data=age_group, x='Age', y='Calories_Burned', color='teal', marker='o')

plt.title('Mean Calories Burned by Age', fontsize=14)
plt.xlabel('Age', fontsize=12)
plt.ylabel('Average Calories Burned', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)

plt.show()

In [ ]:
dfClean['Weight_Bracket'] = pd.cut(dfClean['Weight'], 
                                   bins=[0, 70, 90, 110, 150], 
                                   labels=['<70kg', '70-90kg', '90-110kg', '>110kg'])
dfClean['Age_Bracket'] = pd.cut(dfClean['Age'], bins=[0, 30, 45, 60, 100], labels=['18-30', '31-45', '46-60', '60+'])
dfClean['BMI_Class'] = pd.cut(dfClean['BMI'], bins=[0, 18.5, 25, 30, 100], labels=['Under', 'Normal', 'Over', 'Obese'])


fig, axes = plt.subplots(1, 3, figsize=(20, 6))
teal_style = {'color': 'teal', 'edgecolor': 'black', 'alpha': 0.7}

intensity_means = dfClean.groupby('Exercise_Intensity')['Calories_Burned'].mean()
axes[0].bar(intensity_means.index, intensity_means.values, **teal_style)
axes[0].set_title('Mean Calories by Intensity')
axes[0].set_xlabel('Intensity Level')
axes[0].set_ylabel('Mean Calories')

weight_means = dfClean.groupby('Weight_Bracket')['Calories_Burned'].mean()
axes[1].bar(weight_means.index, weight_means.values, **teal_style)
axes[1].set_title('Mean Calories by Weight Group')
axes[1].set_xlabel('Weight Bracket')

bmi_means = dfClean.groupby('BMI')['Calories_Burned'].mean()
axes[2].bar(dfClean['BMI_Class'].unique(), dfClean.groupby('BMI_Class')['Calories_Burned'].mean(), **teal_style)
axes[2].set_title('Mean Calories by BMI Category')
axes[2].set_xlabel('BMI Category')

plt.suptitle('Weight and Intensity Analysis of Calories Burned (dfClean)', y=1.05, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
numericDf = dfClean.select_dtypes(include='number')
corr = numericDf.corr()
plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask,  annot=True, fmt='.3f', cmap='mako',
            vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 7})
plt.title('Correlation Heatmap')
plt.tight_layout();

plt.show()

In [ ]:
# Train/Test split
# FIX: removed Burn_Rate and Efficiency from TARGET_LEAK_COLS
# (they no longer exist in the DataFrame after Fix 4).
TARGET_LEAK_COLS = {'Calories_Burned'}

numericDf = dfClean.select_dtypes(include='number')
CANDIDATE_FEATURES = [
    c for c in numericDf.columns
    if c not in TARGET_LEAK_COLS
]

X_all = numericDf[CANDIDATE_FEATURES].copy()
y_all = numericDf['Calories_Burned'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=RANDOM_STATE
)

_idx        = y_test.index
y_test_raw  = y_test
y_train_raw = y_train

print(f'Train set : {X_train.shape}')
print(f'Test set  : {X_test.shape}')
print(f'Features  : {X_train.columns.tolist()}')

In [ ]:

# Method 1: Random Forest Importance (fast, purity-based)
print('RF Importance (on training set)')
rf_sel = RandomForestRegressor(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1)
rf_sel.fit(X_train, y_train)
rf_imp = pd.Series(rf_sel.feature_importances_, index=X_train.columns)

# Method 2: Permutation Importance (model-agnostic, unbiased)
print('Permutation Importance (on training set)')
perm = permutation_importance(
    rf_sel, X_train, y_train,
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)
perm_imp = pd.Series(np.clip(perm.importances_mean, 0, None), index=X_train.columns)

def norm01(s): return (s - s.min()) / (s.max() - s.min() + 1e-12)

fs_scores = pd.DataFrame({
    'RF Importance'         : norm01(rf_imp),
    'Permutation Importance': norm01(perm_imp),
})

fs_scores['Combined Score'] = fs_scores.mean(axis=1)
fs_scores = fs_scores.sort_values('Combined Score', ascending=False)

print('\nFeature scores (sorted):')
print(fs_scores.round(3).to_string())

In [ ]:
THRESHOLD = 0.05
important_features = fs_scores[
    fs_scores['Combined Score'] > THRESHOLD
].index.tolist()

assert 'Calories_Burned' not in important_features, 'Target leakage'
assert 'Burn_Rate' not in important_features, 'Target leakage'
assert 'Efficiency' not in important_features, 'Target leakage'

weather_features = ['Weather_Conditions_Rainy', 'Weather_Conditions_Sunny']

selected_features = list(set(important_features + weather_features))

#Redifing the final trtain test data only with data
X_train_final = X_train[selected_features]
X_test_final = X_test[selected_features]

print(f"Features being used for training: {selected_features}")
print(f"Total features: {len(selected_features)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, method in zip(axes, ['RF Importance', 'Permutation Importance', 'Combined Score']):
    top = fs_scores[method].sort_values(ascending=True)
    colors = ['teal' if fs_scores.loc[f,'Combined Score'] > THRESHOLD else 'lightgray'
              for f in top.index]
    ax.barh(top.index, top.values, color=colors)
    ax.set_title(method)
    ax.set_xlabel('Normalised Score')
    ax.axvline(THRESHOLD, color='red', linestyle='--', linewidth=1, label='Threshold')
    ax.legend(fontsize=8)
plt.suptitle('Feature Selection(Training Data)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
X_train = X_train[selected_features]
X_test  = X_test[selected_features]

# Fit scaler on SELECTED TRAINING FEATURES only, so scaler dimensions match the final feature set.
# Tree models: scale-invariant -> use X_train / X_test directly (use_sc=False).
# Ridge, ANN: use X_train_sc / X_test_sc (use_sc=True).
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # fit on training set only
X_test_sc  = scaler.transform(X_test)        # apply same parameters to test

print(f'\nTrain: {X_train.shape}  Test: {X_test.shape}')
print('Scaler fitted on training set only')

In [ ]:
# FIX: Calorie bins updated for new realistic range (80–700 cal)
CALORIE_BINS   = [0, 200, 400, 600, 9999]
CALORIE_LABELS = ['Low', 'Medium', 'High', 'Very High']
results = {}

def _cv_r2(estimator, X, y):
    pred = estimator.predict(X)
    if hasattr(pred, 'flatten'): pred = pred.flatten()
    return r2_score(np.asarray(y), pred)

def evaluate(name, model, Xtr, Xte, ytr, yte_raw, use_sc=False):
    Xtr_cv = Xtr if not use_sc else X_train_sc
    Xte_e  = Xte if not use_sc else X_test_sc

    pred = model.predict(Xte_e)
    if hasattr(pred, 'flatten'): pred = pred.flatten()
    pred = np.clip(pred, 0, None)
    act  = yte_raw.values

    mae  = mean_absolute_error(act, pred)
    rmse = np.sqrt(mean_squared_error(act, pred))
    r2   = r2_score(act, pred)
    mape = np.mean(np.abs((act - pred) / np.clip(act, 1, None))) * 100
    err  = np.abs(act - pred)
    a50  = (err <=  50).mean() * 100
    a100 = (err <= 100).mean() * 100
    a150 = (err <= 150).mean() * 100

    ab = pd.cut(pd.Series(act),  bins=CALORIE_BINS, labels=CALORIE_LABELS).astype(str)
    pb = pd.cut(pd.Series(pred), bins=CALORIE_BINS, labels=CALORIE_LABELS).astype(str)
    f1 = f1_score(ab, pb, average='weighted', zero_division=0)

    cv = cross_val_score(model, Xtr_cv, ytr, cv=5, scoring=_cv_r2, n_jobs=-1)
    results[name] = dict(
        MAE=mae, RMSE=rmse, R2=r2, MAPE=mape,
        **{'Acc@50': a50, 'Acc@100': a100, 'Acc@150': a150,
           'BinnedF1': f1, 'CV_R2': cv.mean(), 'CV_std': cv.std()}
    )
    print(f'  MAE         = {mae:.2f} cal')
    print(f'  RMSE        = {rmse:.2f} cal')
    print(f'  R²          = {r2:.4f}')
    print(f'  MAPE        = {mape:.2f}%')
    print(f'  Acc ±50      = {a50:.1f}%')
    print(f'  Acc ±100     = {a100:.1f}%')
    print(f'  Acc ±150     = {a150:.1f}%')
    print(f'  Binned F1   = {f1:.4f}')
    print(f'  CV R² (orig) = {cv.mean():.4f} ± {cv.std():.4f}')
    return pred

print('evaluate() ready. CALORIE_BINS:', CALORIE_BINS)

In [ ]:

print('Ridge + Polynomial Interactions:')
lr_pipe = Pipeline([
    ('poly',   PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
    ('scaler', StandardScaler()),
    ('ridge',  Ridge(alpha=100.0))
])
lr_pipe.fit(X_train_sc, y_train)

pred_lr = evaluate(
    'Ridge (Poly Interactions)',
    lr_pipe,
    X_train_sc, X_test_sc,
    y_train, y_test_raw,
    use_sc=True
)
lr = lr_pipe


In [ ]:
# Random Forest Regressor
print('Random Forest Regressor:')
rf = RandomForestRegressor(
    n_estimators=300, max_depth=15, min_samples_split=3,
    min_samples_leaf=2, max_features='sqrt',
    random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train, y_train)
pred_rf = evaluate('Random Forest', rf, X_train, X_test, y_train, y_test_raw)


In [ ]:
# XGBoost Regressor
print('XGBoost Regressor:')
_Xtr2, _Xval, _ytr2, _yval = train_test_split(X_train, y_train, test_size=0.15, random_state=RANDOM_STATE)
xgb = XGBRegressor(
    n_estimators=500, learning_rate=0.03, max_depth=4,
    subsample=0.9, colsample_bytree=0.9,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=RANDOM_STATE, verbosity=0
)
xgb.fit(_Xtr2, _ytr2, eval_set=[(_Xval, _yval)], verbose=False)
pred_xgb = evaluate('XGBoost', xgb, X_train, X_test, y_train, y_test_raw)


In [ ]:
# ANN (LeakyReLU + L2 + Huber Loss)
print('ANN (LeakyReLU + L2 + Huber):')
n_f = X_train_sc.shape[1]
REG = l2(1e-4)

ann = Sequential([
    Dense(256, kernel_regularizer=REG, input_shape=(n_f,)), LeakyReLU(0.1),
    BatchNormalization(), Dropout(0.3),
    Dense(128, kernel_regularizer=REG), LeakyReLU(0.1),
    BatchNormalization(), Dropout(0.2),
    Dense(64,  kernel_regularizer=REG), LeakyReLU(0.1),
    BatchNormalization(), Dropout(0.1),
    Dense(32,  kernel_regularizer=REG), LeakyReLU(0.1),
    Dense(1)
])
ann.compile(optimizer=Adam(learning_rate=0.001), loss='huber', metrics=['mae'])

_split = int(len(X_train_sc) * 0.85)
history = ann.fit(
    X_train_sc[:_split], y_train.values[:_split],
    validation_data=(X_train_sc[_split:], y_train.values[_split:]),
    epochs=300, batch_size=32,
    callbacks=[
        EarlyStopping(patience=20, restore_best_weights=True),
        ReduceLROnPlateau(factor=0.5, patience=7, min_lr=1e-6)
    ],
    verbose=0
)

# ANN predict — no expm1, output is raw calories
pred_ann = np.clip(ann.predict(X_test_sc).flatten(), 0, None)
act = y_test_raw.values
err_a = np.abs(act - pred_ann)
ab = pd.cut(pd.Series(act),      bins=CALORIE_BINS, labels=CALORIE_LABELS).astype(str)
pb = pd.cut(pd.Series(pred_ann), bins=CALORIE_BINS, labels=CALORIE_LABELS).astype(str)
results['ANN'] = dict(
    MAE=mean_absolute_error(act, pred_ann),
    RMSE=float(np.sqrt(mean_squared_error(act, pred_ann))),
    R2=r2_score(act, pred_ann),
    MAPE=np.mean(np.abs((act - pred_ann) / np.clip(act, 1, None))) * 100,
    **{'Acc@50':  (err_a <=  50).mean() * 100,
       'Acc@100': (err_a <= 100).mean() * 100,
       'Acc@150': (err_a <= 150).mean() * 100,
       'BinnedF1': f1_score(ab, pb, average='weighted', zero_division=0),
       'CV_R2': float('nan'), 'CV_std': float('nan')}
)
r = results['ANN']
print(f'  MAE={r["MAE"]:.2f}  RMSE={r["RMSE"]:.2f}  R²={r["R2"]:.4f}')
print(f'  Acc±100={r["Acc@100"]:.1f}%  BinnedF1={r["BinnedF1"]:.4f}')

plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'],     label='Train loss')
plt.plot(history.history['val_loss'], label='Val loss')
plt.xlabel('Epoch'); plt.ylabel('Huber Loss'); plt.title('ANN Training')
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# RF Hyperparameter Tuning
rf_param_dist = {
    'n_estimators'     : sp_randint(200, 800),
    'max_depth'        : [8, 10, 12, 15, 20, None],
    'min_samples_split': sp_randint(2, 12),
    'min_samples_leaf' : sp_randint(1, 6),
    'max_features'     : ['sqrt', 'log2', 0.4, 0.6],
    'bootstrap'        : [True, False],
}
rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=rf_param_dist,
    n_iter=60, cv=5, scoring=_cv_r2,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1
)
rf_search.fit(X_train, y_train)
print(f'Best RF params: {rf_search.best_params_}')
print(f'Best CV R²    : {rf_search.best_score_:.4f}')

rf_tuned = RandomForestRegressor(**rf_search.best_params_, random_state=RANDOM_STATE, n_jobs=-1)
rf_tuned.fit(X_train, y_train)
pred_rf_tuned = evaluate('RF (tuned)', rf_tuned, X_train, X_test, y_train, y_test_raw)
print(f'CV R² gain: {results["Random Forest"]["CV_R2"]:.4f} → {results["RF (tuned)"]["CV_R2"]:.4f}')


In [ ]:
# XGBoost Hyperparameter Tuning
xgb_param_dist = {
    'n_estimators'    : sp_randint(200, 700),
    'learning_rate'   : [0.01, 0.02, 0.05, 0.08, 0.1],
    'max_depth'       : sp_randint(3, 8),
    'subsample'       : [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'reg_alpha'       : [0, 0.01, 0.1, 0.5, 1.0],
    'reg_lambda'      : [0.5, 1.0, 2.0, 5.0],
    'min_child_weight': sp_randint(1, 8),
    'gamma'           : [0, 0.1, 0.2, 0.5],
}
xgb_search = RandomizedSearchCV(
    XGBRegressor(random_state=RANDOM_STATE, verbosity=0, n_jobs=-1),
    param_distributions=xgb_param_dist,
    n_iter=60, cv=5, scoring=_cv_r2,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1
)
xgb_search.fit(X_train, y_train)
print(f'Best XGB params: {xgb_search.best_params_}')
print(f'Best CV R²     : {xgb_search.best_score_:.4f}')

xgb_tuned = XGBRegressor(**xgb_search.best_params_, random_state=RANDOM_STATE, verbosity=0)
_Xtr3, _Xv3, _yt3, _yv3 = train_test_split(X_train, y_train, test_size=0.15, random_state=RANDOM_STATE)
xgb_tuned.fit(_Xtr3, _yt3, eval_set=[(_Xv3, _yv3)], verbose=False)
pred_xgb_tuned = evaluate('XGB (tuned)', xgb_tuned, X_train, X_test, y_train, y_test_raw)
print(f'CV R² gain: {results["XGBoost"]["CV_R2"]:.4f} → {results["XGB (tuned)"]["CV_R2"]:.4f}')


In [ ]:
# Ensemble: optimised weighted average of RF (tuned), XGB (tuned), ANN
pred_rf_best  = pred_rf_tuned  if 'RF (tuned)'  in results else pred_rf
pred_xgb_best = pred_xgb_tuned if 'XGB (tuned)' in results else pred_xgb

predMatrix = np.column_stack([pred_rf_best, pred_xgb_best, pred_ann])
actuals    = y_test_raw.values

def _loss(w):
    blendedPred = predMatrix @ w
    ss_res = np.sum((actuals - blendedPred) ** 2)
    ss_tot = np.sum((actuals - actuals.mean()) ** 2)
    return ss_res / ss_tot

optimization = minimize(
    _loss, [1/3, 1/3, 1/3],
    bounds=[(0, 1)] * 3,
    constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1}
)
w_rf, w_xgb, w_ann = optimization.x
print(f'Optimal ensemble weights: RF={w_rf:.3f}  XGB={w_xgb:.3f}  ANN={w_ann:.3f}')

pred_ens = np.clip(w_rf * pred_rf_best + w_xgb * pred_xgb_best + w_ann * pred_ann, 0, None)

act   = y_test_raw.values
err_e = np.abs(act - pred_ens)
abe = pd.cut(pd.Series(act),      bins=CALORIE_BINS, labels=CALORIE_LABELS).astype(str)
pbe = pd.cut(pd.Series(pred_ens), bins=CALORIE_BINS, labels=CALORIE_LABELS).astype(str)

results['Ensemble'] = dict(
    MAE=mean_absolute_error(act, pred_ens),
    RMSE=float(np.sqrt(mean_squared_error(act, pred_ens))),
    R2=r2_score(act, pred_ens),
    MAPE=np.mean(np.abs((act - pred_ens) / np.clip(act, 1, None))) * 100,
    **{'Acc@50':  (err_e <=  50).mean() * 100,
       'Acc@100': (err_e <= 100).mean() * 100,
       'Acc@150': (err_e <= 150).mean() * 100,
       'BinnedF1': f1_score(abe, pbe, average='weighted', zero_division=0),
       'CV_R2': float('nan'), 'CV_std': float('nan')}
)
r = results['Ensemble']
print('\nEnsemble Performance:')
print(f'  MAE={r["MAE"]:.2f}  RMSE={r["RMSE"]:.2f}  R²={r["R2"]:.4f}')
print(f'  Acc±50={r["Acc@50"]:.1f}%  Acc±100={r["Acc@100"]:.1f}%  Acc±150={r["Acc@150"]:.1f}%  BinnedF1={r["BinnedF1"]:.4f}')


In [ ]:
model_order = ['Ridge (Poly Interactions)', 'Random Forest', 'RF (tuned)',
               'XGBoost', 'XGB (tuned)', 'ANN', 'Ensemble']
model_order = [m for m in model_order if m in results]

comp = pd.DataFrame({m: results[m] for m in model_order}).T.reset_index()
comp.rename(columns={'index': 'Model', 'R2': 'R²', 'CV_R2': 'CV R²', 'CV_std': 'CV std'}, inplace=True)
comp = comp.sort_values('R²', ascending=False).reset_index(drop=True)
num_cols = [c for c in comp.columns if c != 'Model']
comp[num_cols] = comp[num_cols].apply(pd.to_numeric, errors='coerce').round(3)

display_cols = ['Model', 'MAE', 'RMSE', 'R²', 'MAPE', 'Acc@50', 'Acc@100', 'Acc@150', 'BinnedF1', 'CV R²']
display_cols = [c for c in display_cols if c in comp.columns]
display(comp[display_cols])

best_model_name = comp.iloc[0]['Model']
print(f'\nBest model: {best_model_name}  '
      f'R²={comp.iloc[0]["R²"]:.4f}  '
      f'Acc±100={comp.iloc[0]["Acc@100"]:.1f}%  '
      f'BinnedF1={comp.iloc[0]["BinnedF1"]:.4f}')


In [ ]:
# Performance bar charts
sns.set_theme(style="white", palette="muted")
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
metrics = ['MAE', 'RMSE', 'R²']
titles  = ['Mean Absolute Error (Lower = Better)',
           'Root Mean Squared Error (Lower = Better)',
           'R² Score (Higher = Better)']

for i, metric in enumerate(metrics):
    df_sorted = comp.sort_values(by=metric, ascending=(metric != 'R²'))
    ax = axes[i]
    bars = ax.bar(df_sorted['Model'], df_sorted[metric],
                  color='teal', edgecolor='black', linewidth=0.5)
    ax.set_title(titles[i], fontsize=12, fontweight='bold', pad=15)
    ax.set_ylabel(metric, fontsize=10)
    ax.tick_params(axis='x', rotation=45, labelsize=9)
    sns.despine(ax=ax)
    ax.yaxis.grid(True, linestyle='--', alpha=0.7)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + h*0.01,
                f'{h:.2f}', ha='center', va='bottom',
                fontsize=9, fontweight='bold', color='#333333')

plt.suptitle('Performance Evaluation', fontsize=16, y=1.05)
plt.tight_layout()
os.makedirs('out/plots', exist_ok=True)
plt.savefig('out/plots/performance.png', dpi=300, bbox_inches='tight')
plt.show()
